# Loan Delinquency Prediction — Model Code
**Atharva Divekar** · Data Scientist I case study

End-to-end, reproducible workflow: load → clean → split → feature engineering → validation & model comparison → holdout evaluation → final model → test scoring.

**How to run:** place `Assignment_Train.xlsx` and `Assignment_Test.xlsx` in `DATA_DIR` (default `../data`) and run all cells top to bottom (≈5 minutes). Every random step uses `RANDOM_STATE = 42`.

**Outputs:**
* `Assignment_Test_Atharva_Divekar.csv` — the required file: all 93 original test columns unchanged and in original row order, the 21 model inputs (prefix `fe__`), `risk_score` (1–100, 100 = highest relative risk) and `risk_tier`.
* `Assignment_Test_Atharva_Divekar_scores_only.csv` — convenience summary with `UID`, `risk_score`, `risk_tier` only (same rows and order).

**Environment used:** Python 3.14.7, pandas 3.0.6, numpy 2.5.4, scikit-learn 1.9.1, scipy 1.18.1, catboost 1.2.10 (CatBoost is only used in the model comparison, not in the final model).

In [ ]:
import warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_colwidth", None)

RANDOM_STATE = 42
DATA_DIR = "../data"
OUTPUT_CSV = "Assignment_Test_Atharva_Divekar.csv"
SCORES_ONLY_CSV = "Assignment_Test_Atharva_Divekar_scores_only.csv"

## 1. Load data
The literal text `"NaT"` is used throughout the files as a missing-value placeholder, so both files are loaded with `na_values=["NaT"]` for modelling.
A separate, untouched copy of the test file (`dtype=object`) is kept so the output CSV reproduces the original columns exactly.

In [ ]:
train_raw = pd.read_excel(f"{DATA_DIR}/Assignment_Train.xlsx", na_values=["NaT"])
test_raw  = pd.read_excel(f"{DATA_DIR}/Assignment_Test.xlsx",  na_values=["NaT"])
test_original = pd.read_excel(f"{DATA_DIR}/Assignment_Test.xlsx", dtype=object)   # for the output file only

assert train_raw.shape == (10_000, 94) and test_raw.shape == (5_000, 93)
assert list(test_raw.columns) == [c for c in train_raw.columns if c != "Target"]
assert train_raw["UID"].is_unique and test_raw["UID"].is_unique
assert (test_original["UID"].values == test_raw["UID"].values).all()
print("train:", train_raw.shape, "| test:", test_raw.shape)

## 2. Target
`Target` is 0/1 except 8 rows whose label is the text "NaT" (now missing). These rows carry no supervised signal and are dropped; the target is never imputed.
Assumption: `Target = 1` means delinquent (minority class, ≈8.8%).

In [ ]:
has_label = train_raw["Target"].notna()
train = train_raw.loc[has_label].copy()
train["Target"] = train["Target"].astype("int8")

assert len(train) == 9_992 and set(train["Target"].unique()) == {0, 1}
print("rows dropped:", int((~has_label).sum()), "| positives:", int(train["Target"].sum()),
      "| positive rate:", round(train["Target"].mean(), 4))

## 2b. Data checks (evidence for the data-quality findings in the report)
Target-free checks on the labelled training data, plus train-vs-test comparisons. Nothing here changes the data used later.

In [ ]:
feature_cols = [c for c in train.columns if c not in ("UID", "Target")]

# 1. "NaT" text placeholder: columns that are mixed number/text when loaded WITHOUT na_values
raw_default = pd.read_excel(f"{DATA_DIR}/Assignment_Train.xlsx")
mixed = raw_default[feature_cols].select_dtypes(include="object", exclude="str").columns
print("mixed-type feature columns without na_values=['NaT']:", len(mixed))

# 2. -1 sentinel: columns where more than half the values are -1 (yet 0% missing to pandas)
minus1 = (train[feature_cols].select_dtypes("number").eq(-1).mean() * 100).round(1)
display(minus1[minus1 > 50].sort_values(ascending=False).to_frame("% equal to -1"))

# 3. Constant / empty columns (at most one known value)
print("constant / empty columns:", [c for c in feature_cols if train[c].nunique() <= 1])

# 4. Exact duplicates across all features (UID and Target excluded)
print("exact duplicate rows:", int(train.duplicated(subset=feature_cols, keep=False).sum()))

In [ ]:
# 5. Truncated phone numbers: real numbers end in many zeros, so different people share a value
phone = train["Email Name Attributes.phoneNumber"]
real_phone = phone[phone != -1].astype("int64").astype(str)
trailing_zeros = real_phone.str.len() - real_phone.str.rstrip("0").str.len()
print(f"phone numbers present: {len(real_phone)} | distinct: {real_phone.nunique()} | "
      f"ending in >= 6 zeros: {(trailing_zeros >= 6).mean():.1%}")

# 6. Repeat applicants: same name + UPI name, then also same region and exact phone digital age
norm = lambda s: s.astype(str).str.lower().str.split().str.join(" ")
has_upi = train["Phone to Name.UPIname"].notna()
keys = pd.DataFrame({"name": norm(train["name"]), "upi": norm(train["Phone to Name.UPIname"]),
                     "region": train["Phone Network.currentNetworkRegion"],
                     "phone_age": train["Phone Name Attributes.digitalage"]})[has_upi]
print("rows sharing name + UPI name:", int(keys.duplicated(["name", "upi"], keep=False).sum()),
      "| ... + region + exact phone age:", int(keys.duplicated(["name", "upi", "region", "phone_age"], keep=False).sum()))

In [ ]:
# 7. Train vs test: operator mix, billing type missing by operator, latest revocation month
net, bill = "Phone Network.currentNetworkName", "Phone Network.numberBillingType"
display(pd.DataFrame({"train": train[net].value_counts(normalize=True),
                      "test":  test_raw[net].value_counts(normalize=True)}).mul(100).round(1).rename_axis("operator (%)"))
print("billing type missing: train", f"{train[bill].isna().mean():.1%}", "| test", f"{test_raw[bill].isna().mean():.1%}")
display(pd.DataFrame({"train": train.groupby(net)[bill].apply(lambda s: s.isna().mean()),
                      "test":  test_raw.groupby(net)[bill].apply(lambda s: s.isna().mean())}).mul(100).round(0)
        .rename_axis("billing missing by operator (%)"))
rev = "Phone Name Attributes.revocationMonth"
print("latest revocation month: train", pd.to_datetime(train[rev]).max().date(),
      "| test", pd.to_datetime(test_raw[rev]).max().date())

## 3. Train / holdout split
Stratified 70:30 split made before any target-based analysis. `dev` is used for all exploration, feature and model decisions (via 5-fold CV); `holdout` is kept untouched for the final estimate.

In [ ]:
dev, holdout = train_test_split(train, test_size=0.30, stratify=train["Target"], random_state=RANDOM_STATE)
assert len(dev) == 6_994 and len(holdout) == 2_998 and set(dev["UID"]).isdisjoint(holdout["UID"])
y_dev, y_hold = dev["Target"], holdout["Target"]
print("dev:", dev.shape, round(y_dev.mean(), 4), "| holdout:", holdout.shape, round(y_hold.mean(), 4))

## 4. Feature engineering
All steps use only each row's own values (nothing is learned from other rows or the target), so they apply identically to dev, holdout and test.
1. **Sentinels** — `-1` means "not found in the vendor network". Flags are created first, then `-1 → NaN` in four columns.
2. **Platform shares** — share of known platforms (email / phone) on which the person is registered (paytm and skype excluded as ~96–100% missing).
3. **Ordinal encoding** — footprint levels mapped to ordered integers; `NO_RECORDS` becomes NaN plus its own flag.
4. **Categories** — fixed level lists chosen from dev; rare levels → `other`; billing keeps `unknown` and `missing` separately.

In [ ]:
SENTINEL_COLS = [
    "Email Attributes.digitalAge", "Email Name Attributes.digitalage",
    "Phone Name Attributes.phoneNameDigitalAge", "Email Name Attributes.nameMatchScore",
]
NOT_FOUND_FLAGS = {                                   # flag name -> column it is built from
    "email_name_not_found": "Email Name Attributes.nameMatchScore",
    "email_not_found":      "Email Name Attributes.digitalage",
    "phone_name_not_found": "Phone Name Attributes.phoneNameDigitalAge",
}

def handle_sentinels(data, mode="flag"):
    """mode='raw': leave -1 as-is. mode='flag': add not-found flags, then -1 -> NaN."""
    out = data.copy()
    if mode == "raw":
        return out
    if mode != "flag":
        raise ValueError(f"unknown mode: {mode}")
    for flag, source in NOT_FOUND_FLAGS.items():          # flags first, while the -1 values still exist
        out[flag] = out[source].eq(-1).astype("int8")
    out[SENTINEL_COLS] = out[SENTINEL_COLS].replace(-1, np.nan)
    return out

def add_platform_share(data):
    """Share of known platforms where the person is registered, separately for email and phone."""
    out = data.copy()
    for source in ["Email", "Phone"]:
        cols = [c for c in data.columns
                if c.startswith(f"{source} Social Advance.") and not c.endswith((".paytm", ".skype"))]
        n_known   = data[cols].notna().sum(axis=1)
        n_present = data[cols].sum(axis=1)
        out[f"{source.lower()}_platform_known"] = n_known
        out[f"{source.lower()}_platform_share"] = n_present / n_known.replace(0, np.nan)
    return out

ORDINAL_MAPS = {
    "Email Attributes.footprint": {"LOW": 0, "MEDIUM": 1, "HIGH": 2, "NO_RECORDS": np.nan},
    "Phone Name Attributes.phoneFootprintStrengthOverall": {"Low": 0, "Medium": 1, "High": 2, "Very High": 3},
    "Email Name Attributes.emailFootprintStrength": {"Very Low": 0, "Low": 1, "High": 2, "Very High": 3},
}

def encode_ordinals(data, ordinal_maps=ORDINAL_MAPS):
    """Convert ordinal string values to integers, using the provided mapping."""
    out = data.copy()
    out["footprint_no_records"] = out["Email Attributes.footprint"].eq("NO_RECORDS").astype("int8")
    for col, mapping in ordinal_maps.items():
        out[col] = out[col].map(mapping)
    return out

CATEGORY_LEVELS = {
    # column: (levels to keep, keep missing as its own level?)
    "Phone Network.numberBillingType":           (["prepaid", "postpaid", "unknown"], True),
    "Phone Network.currentNetworkName":          (["Airtel", "Reliance Jio"],         False),
    "Email Attributes.emailFinalRecommendation": (["Good", "Check"],                  False),
}

def clean_categories(data, levels=CATEGORY_LEVELS):
    """Keep listed levels; everything else becomes 'other' (missing becomes 'missing' if requested)."""
    out = data.copy()
    for col, (keep, missing_separate) in levels.items():
        s = out[col]
        cleaned = s.where(s.isin(keep), "other")
        if missing_separate:
            cleaned = cleaned.where(s.notna(), "missing")
        out[col] = cleaned
    return out

NUMERIC_FEATURES = [
    "Email Attributes.digitalAge", "Email Name Attributes.digitalage",
    "Phone Name Attributes.phoneNameDigitalAge", "Email Name Attributes.nameMatchScore",
    "Email Name Attributes.nameEmailMatch", "Email Attributes.uniqueHits",
    "email_platform_share", "phone_platform_share",
    "Email Attributes.footprint", "Phone Name Attributes.phoneFootprintStrengthOverall",
    "Email Name Attributes.emailFootprintStrength",
]
FLAG_FEATURES = [
    "Phone Social Advance.whatsapp", "Phone Network.numberHasPortingHistory",
    "Email Name Attributes.firstNameMatch", "Email Name Attributes.lastNameMatch",
    "email_name_not_found", "email_not_found", "phone_name_not_found", "footprint_no_records",
]
CATEGORICAL_FEATURES = [
    "Phone Network.numberBillingType", "Phone Network.currentNetworkName",
    "Email Attributes.emailFinalRecommendation",
]
MODEL_FEATURES = NUMERIC_FEATURES + FLAG_FEATURES + CATEGORICAL_FEATURES      # the 22 selected features

def feature_groups(sentinel_mode="flag"):
    flags = FLAG_FEATURES if sentinel_mode == "flag" else [f for f in FLAG_FEATURES if f not in NOT_FOUND_FLAGS]
    return list(NUMERIC_FEATURES), flags, list(CATEGORICAL_FEATURES)

def build_features(data, sentinel_mode="flag"):
    """Apply all feature steps in order and return UID + model features (+ Target if present)."""
    out = clean_categories(encode_ordinals(add_platform_share(handle_sentinels(data, mode=sentinel_mode))))
    numeric, flags, categorical = feature_groups(sentinel_mode)
    keep = ["UID"] + numeric + flags + categorical + (["Target"] if "Target" in out.columns else [])
    return out[keep]

def fill_missing_text(X, cats):
    """Missing text -> 'missing' (row-wise, nothing learned)."""
    X = X.copy()
    X[cats] = X[cats].astype("object").where(X[cats].notna(), "missing")
    return X

In [ ]:
num_sel, flag_sel, cat_sel = feature_groups("flag")
dev_fe, hold_fe, test_fe = build_features(dev), build_features(holdout), build_features(test_raw)

assert list(hold_fe.columns) == list(dev_fe.columns)
assert list(test_fe.columns) == [c for c in dev_fe.columns if c != "Target"]
for raw, fe in [(dev, dev_fe), (holdout, hold_fe), (test_raw, test_fe)]:
    assert len(fe) == len(raw) and (fe["UID"].values == raw["UID"].values).all()   # rows kept, same order
assert not dev_fe[NUMERIC_FEATURES].eq(-1).any().any()

X_dev   = fill_missing_text(dev_fe.drop(columns=["UID", "Target"]), cat_sel)
X_hold  = fill_missing_text(hold_fe.drop(columns=["UID", "Target"]), cat_sel)
X_raw   = fill_missing_text(build_features(dev, "raw").drop(columns=["UID", "Target"]), cat_sel)
print("not-found flag shares (dev):", dev_fe[list(NOT_FOUND_FLAGS)].mean().round(3).to_dict())
print("feature table:", X_dev.shape)

## 5. Validation design and model comparison
* **Validation:** stratified 5-fold CV on dev (≈123 delinquents per validation fold), identical folds for every comparison.
* **Metrics:** PR-AUC (primary — focuses on the rare class; random ≈ 0.088) and ROC-AUC. Both measure ranking and need no threshold.
* **Decision rule (fixed in advance):** a change is adopted only if it improves PR-AUC by more than ≈0.03 (the fold-to-fold standard deviation); when results are tied, the simpler / more interpretable option is kept.
* Preprocessing (imputation, scaling, one-hot encoding) sits inside the model pipeline, so it is learned from training folds only.

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def make_model(kind, numeric, categorical):
    """One untuned model of the given kind, with preprocessing suited to it."""
    if kind == "catboost":
        return CatBoostClassifier(iterations=500, random_seed=RANDOM_STATE, verbose=0,
                                  cat_features=categorical, allow_writing_files=False)
    num_steps = [] if kind == "hgb" else [("impute", SimpleImputer(strategy="median"))]
    if kind == "logreg":
        num_steps.append(("scale", StandardScaler()))
    prep = ColumnTransformer([
        ("num", Pipeline(num_steps) if num_steps else "passthrough", numeric),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20), categorical),
    ])
    model = {
        "logreg": LogisticRegression(max_iter=2000),
        "rf":     RandomForestClassifier(n_estimators=500, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE),
        "hgb":    HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    }[kind]
    return Pipeline([("prep", prep), ("model", model)])

def cv_evaluate(kind, X, y, numeric, categorical, params=None):
    """Manual 5-fold CV (a fresh model per fold). Returns metrics and out-of-fold predictions."""
    oof = pd.Series(np.nan, index=X.index)
    pr, roc = [], []
    for train_idx, val_idx in CV.split(X, y):
        model = make_model(kind, numeric, categorical)
        if params:
            model.set_params(**params)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        p = model.predict_proba(X.iloc[val_idx])[:, 1]
        oof.iloc[val_idx] = p
        pr.append(average_precision_score(y.iloc[val_idx], p))
        roc.append(roc_auc_score(y.iloc[val_idx], p))
    return {"pr_auc_mean": np.mean(pr), "pr_auc_std": np.std(pr),
            "roc_auc_mean": np.mean(roc), "roc_auc_std": np.std(roc)}, oof

def run_variant(drop=(), add_region=False):
    """Logistic regression on the 22 features minus `drop`, optionally plus raw region."""
    numeric     = [c for c in num_sel + flag_sel if c not in drop]
    categorical = [c for c in cat_sel if c not in drop]
    X = X_dev[numeric + categorical].copy()
    if add_region:
        X["Phone Network.currentNetworkRegion"] = dev["Phone Network.currentNetworkRegion"]
        categorical = categorical + ["Phone Network.currentNetworkRegion"]
    return cv_evaluate("logreg", fill_missing_text(X, categorical), y_dev, numeric, categorical)[0]

### 5.1 Baseline and the `-1` handling decision
Dummy model (predicts the base rate) as the floor; logistic regression with `-1` left as-is (raw) vs `-1 → NaN` + not-found flags.

In [ ]:
res = {}
dummy_pr  =[average_precision_score(y_dev.iloc[v], np.full(len(v), y_dev.iloc[t].mean())) for t, v in CV.split(X_dev, y_dev)]
res["dummy (base rate)"] = {"pr_auc_mean": np.mean(dummy_pr), "pr_auc_std": np.std(dummy_pr), "roc_auc_mean": 0.5, "roc_auc_std": 0.0}
num_raw, flag_raw, _ = feature_groups("raw")
res["logreg, -1 raw"], _      = cv_evaluate("logreg", X_raw, y_dev, num_raw + flag_raw, cat_sel)
res["logreg, -1 -> NaN + flags"], oof_lr22 = cv_evaluate("logreg", X_dev, y_dev, num_sel + flag_sel, cat_sel)
pd.DataFrame(res).T.round(3)

### 5.2 Model type × feature set
"All columns" = the 22 engineered/selected features plus every other usable column, excluding identifiers/PII (`UID`, `name`, `UPIname`, `phoneNumber`), region (fairness), constant columns (decided from dev) and `revocationMonth` (a date).

In [ ]:
FIXED_EXCLUDE = ["UID", "Target", "name", "Phone to Name.UPIname", "Email Name Attributes.phoneNumber",
                 "Phone Network.currentNetworkRegion", "Phone Network.portedFromNetworkRegion",
                 "Phone Name Attributes.revocationMonth"]
CONSTANT_COLS = [c for c in dev.columns if dev[c].nunique() <= 1]

def build_all_columns(data):
    out = clean_categories(encode_ordinals(add_platform_share(handle_sentinels(data, "flag"))))
    return out.drop(columns=[c for c in FIXED_EXCLUDE + CONSTANT_COLS if c in out.columns])

X_all_raw = build_all_columns(dev)
CAT_ALL = X_all_raw.select_dtypes(include=["str", "object"]).columns.tolist()
NUM_ALL = [c for c in X_all_raw.columns if c not in CAT_ALL]
X_all = fill_missing_text(X_all_raw, CAT_ALL)
assert set(MODEL_FEATURES) <= set(X_all.columns)
print("all-columns table:", X_all.shape)

FEATURE_SETS = {"selected 22": (X_dev, num_sel + flag_sel, cat_sel), "all usable": (X_all, NUM_ALL, CAT_ALL)}
grid_rows, oof_preds = [], {}
for kind in ["logreg", "rf", "hgb", "catboost"]:
    for set_name, (X, numeric, categorical) in FEATURE_SETS.items():
        metrics, oof = cv_evaluate(kind, X, y_dev, numeric, categorical)
        grid_rows.append({"model": kind, "features": set_name, **metrics})
        oof_preds[(kind, set_name)] = oof
grid = pd.DataFrame(grid_rows).round(3)
grid.sort_values("pr_auc_mean", ascending=False)

### 5.3 Permutation importance (Random Forest, all columns)
Drop in validation PR-AUC when a column is shuffled, averaged over the 5 folds. Used for insight only; candidates it suggests are tested as a group below.

In [ ]:
fold_drops = []
for train_idx, val_idx in CV.split(X_all, y_dev):
    rf_model = make_model("rf", NUM_ALL, CAT_ALL).fit(X_all.iloc[train_idx], y_dev.iloc[train_idx])
    r = permutation_importance(rf_model, X_all.iloc[val_idx], y_dev.iloc[val_idx],
                               scoring="average_precision", n_repeats=5, random_state=RANDOM_STATE)
    fold_drops.append(pd.Series(r.importances_mean, index=X_all.columns))
drops = pd.concat(fold_drops, axis=1)
importance = pd.DataFrame({"pr_auc_drop": drops.mean(axis=1), "std_over_folds": drops.std(axis=1),
                           "folds_positive": (drops > 0).sum(axis=1),
                           "in_selected_22": drops.index.isin(MODEL_FEATURES)}
                          ).sort_values("pr_auc_drop", ascending=False).round(4)
display(importance.head(15))

CANDIDATES = ["Phone Social Advance.instagram", "Email Social Advance.amazon", "email_platform_known",
              "Mobile GIN.l2Confidence", "Mobile GIN.l1Confidence", "Phone Social Advance.housing"]
m_plus, _ = cv_evaluate("logreg", fill_missing_text(X_all_raw[MODEL_FEATURES + CANDIDATES], cat_sel),
                        y_dev, num_sel + flag_sel + CANDIDATES, cat_sel)
pd.DataFrame({"selected 22": res["logreg, -1 -> NaN + flags"], "22 + 6 candidates": m_plus}).T.round(3)

### 5.4 Class weighting, sensitivity checks, ablation, network
* **Class weighting** — `class_weight="balanced"` vs none.
* **Sensitivity** — WhatsApp and billing type could reflect events after default if the vendor data was pulled late (collection date unknown); region tested for the fairness decision.
* **Ablation** — contribution of each engineered feature group.
* **Network** — the training data is 94% Airtel while test is 45% Airtel, so the need for `currentNetworkName` is tested.

In [ ]:
m_bal, _ = cv_evaluate("logreg", X_dev, y_dev, num_sel + flag_sel, cat_sel, params={"model__class_weight": "balanced"})
display(pd.DataFrame({"no weighting": res["logreg, -1 -> NaN + flags"], "class_weight='balanced'": m_bal}).T.round(3))

WHATSAPP, BILLING, NETWORK = "Phone Social Advance.whatsapp", "Phone Network.numberBillingType", "Phone Network.currentNetworkName"
SHARES    = ["email_platform_share", "phone_platform_share"]
NOT_FOUND = list(NOT_FOUND_FLAGS)
ORDINALS  = list(ORDINAL_MAPS) + ["footprint_no_records"]
variants = {
    "baseline (22 features)":     run_variant(),
    "without WhatsApp":           run_variant(drop=[WHATSAPP]),
    "without billing":            run_variant(drop=[BILLING]),
    "without both":               run_variant(drop=[WHATSAPP, BILLING]),
    "with region":                run_variant(add_region=True),
    "without platform shares":    run_variant(drop=SHARES),
    "without not-found flags":    run_variant(drop=NOT_FOUND),
    "without footprint ordinals": run_variant(drop=ORDINALS),
    "without all engineered":     run_variant(drop=SHARES + NOT_FOUND + ORDINALS),
    "without network (final)":    run_variant(drop=[NETWORK]),
}
checks = pd.DataFrame(variants).T.round(3)
checks["pr_auc_change"] = (checks["pr_auc_mean"] - checks.loc["baseline (22 features)", "pr_auc_mean"]).round(3)
checks

### 5.5 Ensemble check
Rank-average of logistic regression and the best tree model (Random Forest), compared fold by fold.

In [ ]:
lr_oof, rf_oof = oof_preds[("logreg", "selected 22")], oof_preds[("rf", "selected 22")]
print("rank correlation LR vs RF:", round(spearmanr(lr_oof, rf_oof).statistic, 3))
ens_oof = (lr_oof.rank(pct=True) + rf_oof.rank(pct=True)) / 2
fold_rows = []
for f, (_, val) in enumerate(CV.split(X_dev, y_dev), 1):
    lr_s, en_s = average_precision_score(y_dev.iloc[val], lr_oof.iloc[val]), average_precision_score(y_dev.iloc[val], ens_oof.iloc[val])
    fold_rows.append({"fold": f, "logreg": lr_s, "rank_average": en_s, "difference": en_s - lr_s})
pd.DataFrame(fold_rows).round(3)

## 6. Final model and holdout evaluation
**Final model:** logistic regression (no class weighting) on **21 features** — the 22 selected features minus `currentNetworkName`.
Tier cut-offs are fixed from dev out-of-fold scores **before** scoring the holdout: decline = top 5%, review = next 15%, approve = rest.

In [ ]:
cat_final  = [c for c in cat_sel if c != NETWORK]
FINAL_COLS = (num_sel + flag_sel, cat_final)
FINAL_FEATURES = num_sel + flag_sel + cat_final
assert len(FINAL_FEATURES) == 21

final_cv, oof_final_dev = cv_evaluate("logreg", X_dev, y_dev, *FINAL_COLS)
cut_decline, cut_review = oof_final_dev.quantile(0.95), oof_final_dev.quantile(0.80)

model_dev = make_model("logreg", *FINAL_COLS).fit(X_dev, y_dev)
p_hold = pd.Series(model_dev.predict_proba(X_hold)[:, 1], index=X_hold.index)
holdout_pr, holdout_roc = average_precision_score(y_hold, p_hold), roc_auc_score(y_hold, p_hold)
print(f"dev CV PR-AUC {final_cv['pr_auc_mean']:.3f} ± {final_cv['pr_auc_std']:.3f} | "
      f"HOLDOUT PR-AUC {holdout_pr:.3f} | HOLDOUT ROC-AUC {holdout_roc:.3f}")

hold_tier = pd.Series(np.select([p_hold >= cut_decline, p_hold >= cut_review], ["1 decline", "2 review"], "3 approve"),
                      index=p_hold.index, name="tier")
outcome = pd.crosstab(hold_tier, y_hold.rename("delinquent"), margins=True)
outcome["delinquency_rate"] = (outcome[1] / outcome["All"]).round(3)
outcome

In [ ]:
# Coefficients of the final model (fit on dev). Numeric inputs are standardised, so each coefficient is per one SD.
names = model_dev.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(model_dev.named_steps["model"].coef_[0], index=names)
pd.DataFrame({"coef": coefs, "odds_ratio": np.exp(coefs)}).sort_values("coef", ascending=False).round(3)

### Performance by operator (out-of-fold on all labelled rows)
The training data is 94% Airtel; ranking quality is checked separately for non-Airtel customers (ROC-AUC with a bootstrap 95% interval).

In [ ]:
X_train = fill_missing_text(build_features(train).drop(columns=["UID", "Target"]), cat_sel)
y_train = train["Target"]

reference_oof = pd.Series(np.nan, index=X_train.index)
for train_idx, val_idx in CV.split(X_train, y_train):
    fold_model = make_model("logreg", *FINAL_COLS).fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
    reference_oof.iloc[val_idx] = fold_model.predict_proba(X_train.iloc[val_idx])[:, 1]

network_group = train[NETWORK].where(train[NETWORK] == "Airtel", "non-Airtel")
rng = np.random.default_rng(RANDOM_STATE)
rows = []
for group in ["Airtel", "non-Airtel"]:
    m = network_group == group
    yg, pg = y_train[m].to_numpy(), reference_oof[m].to_numpy()
    boot = []
    for _ in range(500):
        i = rng.integers(0, len(yg), len(yg))
        if yg[i].min() != yg[i].max():
            boot.append(roc_auc_score(yg[i], pg[i]))
    rows.append({"group": group, "people": int(m.sum()), "delinquents": int(yg.sum()), "base_rate": yg.mean(),
                 "roc_auc": roc_auc_score(yg, pg), "roc_auc_95ci": "{:.2f} - {:.2f}".format(*np.percentile(boot, [2.5, 97.5]))})
pd.DataFrame(rows).round(3)

## 7. Final fit and test scoring
* The final model is refit on **all 9,992 labelled rows** (dev + holdout).
* **Risk score (1–100):** the percentile of an applicant's predicted risk within a fixed reference distribution — the out-of-fold predictions for all labelled rows computed above. `score = ceil(100 × share of reference predictions ≤ p)`, clipped to 1–100. A score of 96 means *riskier than ~95% of the reference population*. It is a **relative-risk ranking, not a calibrated probability**, and an applicant's score does not depend on who else is scored.
* **Tiers:** decline ≥ 96 · review 81–95 · approve ≤ 80 (5% / 15% / 80% of the reference population). The cut-offs are a policy choice under illustrative costs and should be set by the lender from its own loss, margin and review capacity.

In [ ]:
REFERENCE_SORTED = np.sort(reference_oof.values)

def risk_score(p):
    """1-100 score = percentile of predicted risk within the reference distribution. 100 = highest relative risk."""
    share_at_or_below = np.searchsorted(REFERENCE_SORTED, p, side="right") / len(REFERENCE_SORTED)
    return np.clip(np.ceil(share_at_or_below * 100), 1, 100).astype(int)

def risk_tier(score):
    return np.select([score >= 96, score >= 81], ["decline", "review"], "approve")

final_model = make_model("logreg", *FINAL_COLS).fit(X_train, y_train)

X_test = fill_missing_text(test_fe.drop(columns=["UID"]), cat_sel)
p_test = final_model.predict_proba(X_test)[:, 1]
test_score = risk_score(p_test)

assert len(test_score) == 5_000 and test_score.min() >= 1 and test_score.max() <= 100
assert np.all(np.diff(risk_score(np.sort(p_test))) >= 0)              # higher risk never gets a lower score
print("reference share with score >= 96:", round((risk_score(reference_oof.values) >= 96).mean(), 3))
pd.Series(risk_tier(test_score)).value_counts(normalize=True).round(3)

In [ ]:
# Output: original columns exactly as supplied + the 21 model inputs (fe__) + risk_score + risk_tier
engineered = test_fe[FINAL_FEATURES].add_prefix("fe__").reset_index(drop=True)
submission = pd.concat([test_original.reset_index(drop=True), engineered], axis=1)
submission["risk_score"] = test_score
submission["risk_tier"]  = risk_tier(test_score)

# Checks required by the brief
assert len(submission) == len(test_original) == 5_000                              # no rows dropped or duplicated
assert submission["UID"].is_unique
assert (submission["UID"].values == test_original["UID"].values).all()             # original row order
assert list(submission.columns[:93]) == list(test_original.columns)                 # original columns first, unchanged order
assert submission[test_original.columns].equals(test_original)                     # original values untouched
assert submission["risk_score"].notna().all() and submission["risk_score"].between(1, 100).all()

submission.to_csv(OUTPUT_CSV, index=False)
check = pd.read_csv(OUTPUT_CSV)
assert check.shape == submission.shape and (check["UID"].values == test_original["UID"].values).all()
print("written:", OUTPUT_CSV, check.shape)

# Convenience summary (not a replacement for the file above): UID is enough for traceability, so no names
scores_only = submission[["UID", "risk_score", "risk_tier"]]
scores_only.to_csv(SCORES_ONLY_CSV, index=False)
assert (pd.read_csv(SCORES_ONLY_CSV)["UID"].values == test_original["UID"].values).all()
print("written:", SCORES_ONLY_CSV, scores_only.shape)

In [ ]:
summary = pd.Series({
    "dev CV PR-AUC (final, 21 features)": f"{final_cv['pr_auc_mean']:.3f} ± {final_cv['pr_auc_std']:.3f}",
    "dev CV ROC-AUC (final)":             f"{final_cv['roc_auc_mean']:.3f} ± {final_cv['roc_auc_std']:.3f}",
    "holdout PR-AUC":                     f"{holdout_pr:.3f}  (random = {y_hold.mean():.3f})",
    "holdout ROC-AUC":                    f"{holdout_roc:.3f}",
    "test rows scored":                   len(submission),
    "test share decline / review / approve": " / ".join(f"{(submission['risk_tier'] == t).mean():.1%}" for t in ["decline", "review", "approve"]),
})
summary.to_frame("value")